# SAM 2 Auto-Labeling for 2000 Building Defect Images
### Updated Version: Processes 2000 images, splits into 2 datasets, immediate download

**Features:**
- ✅ Processes exactly 2000 images (configurable)
- ✅ Splits dataset into 2 folders (1000 images each)
- ✅ Immediate download after completion
- ✅ YOLO format labels ready for training
- ✅ Fully automatic - no manual labeling

**Expected Time:**
- 2000 images on T4 GPU: ~3-4 hours
- Automatic download at the end

**Output Structure:**
```
sam2_labeled_results/
├── batch_1/
│   ├── images/  (1000 images)
│   └── labels/  (1000 .txt files)
└── batch_2/
    ├── images/  (1000 images)
    └── labels/  (1000 .txt files)
```

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 1: Configuration - SET YOUR PARAMETERS HERE
# ═══════════════════════════════════════════════════════════════

# CONFIGURATION
MAX_IMAGES = 2000  # Total images to process
IMAGES_PER_BATCH = 1000  # Split into batches of 1000
NUM_BATCHES = 2  # Number of batches (2000 / 1000 = 2)

# ZIP file location in Google Drive
ZIP_PATHS = [
    '/content/drive/MyDrive/SAM2_AutoLabel/filtered_images.zip',
    '/content/drive/MyDrive/filtered_images.zip',
    '/content/filtered_images.zip'
]

# Output directories
OUTPUT_DIR = '/content/sam2_labeled_results'
DOWNLOAD_IMMEDIATELY = True  # Auto-download when complete

print("✅ Configuration loaded")
print(f"   Total images to process: {MAX_IMAGES:,}")
print(f"   Images per batch: {IMAGES_PER_BATCH:,}")
print(f"   Number of batches: {NUM_BATCHES}")
print(f"   Auto-download: {DOWNLOAD_IMMEDIATELY}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 2: Setup - Mount Drive & Install Dependencies
# ═══════════════════════════════════════════════════════════════

from google.colab import drive, files
import os

# Mount Google Drive
print("📂 Mounting Google Drive...")
drive.mount('/content/drive')
print("✅ Google Drive mounted\n")

# Install SAM 2 and dependencies
print("📦 Installing SAM 2 and dependencies...")
!pip install git+https://github.com/facebookresearch/segment-anything-2.git -q
!pip install opencv-python pillow numpy tqdm -q

print("\n✅ Installation complete!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 3: Load SAM 2 Model
# ═══════════════════════════════════════════════════════════════

import torch
import numpy as np
from PIL import Image
import cv2
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

print("="*70)
print("🖥️  Hardware Check")
print("="*70)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device.upper()}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  WARNING: No GPU! Processing will be VERY slow.")
    print("Fix: Runtime → Change runtime type → T4 GPU")

print("\n" + "="*70)
print("📥 Loading SAM 2 Model")
print("="*70)

# Download SAM 2 checkpoint
!wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt

# Load model
sam2_checkpoint = "sam2_hiera_large.pt"
model_cfg = "sam2_hiera_l.yaml"

sam2 = build_sam2(model_cfg, sam2_checkpoint, device=device)

# Create automatic mask generator
mask_generator = SAM2AutomaticMaskGenerator(
    model=sam2,
    points_per_side=32,
    points_per_batch=64,
    pred_iou_thresh=0.88,
    stability_score_thresh=0.95,
    stability_score_offset=1.0,
    crop_n_layers=1,
    box_nms_thresh=0.7,
    crop_n_points_downscale_factor=2,
    min_mask_region_area=100,
)

print("\n✅ SAM 2 Automatic Mask Generator Ready!")
print("   This will automatically segment ALL objects in images")
print("   No manual prompts needed!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 4: Define Defect Classes & Classification
# ═══════════════════════════════════════════════════════════════

# 15 Building Defect Classes (same as your YOLO model)
DEFECT_CLASSES = {
    0: 'general_damage',
    1: 'cracks',
    2: 'mold',
    3: 'water_damage',
    4: 'structural_damage',
    5: 'electrical_issues',
    6: 'plumbing_issues',
    7: 'roofing_damage',
    8: 'window_damage',
    9: 'door_damage',
    10: 'floor_damage',
    11: 'wall_damage',
    12: 'ceiling_damage',
    13: 'hvac_issues',
    14: 'insulation_issues'
}

CLASS_NAMES = list(DEFECT_CLASSES.values())

def classify_segment_heuristic(mask, image):
    """
    Classify a segment using heuristics based on:
    - Color features
    - Shape features (aspect ratio, area)
    - Texture features
    
    Returns: (class_id, confidence)
    """
    segment_pixels = image[mask]
    
    if len(segment_pixels) == 0:
        return None, 0.0
    
    # Calculate color features
    mean_color = segment_pixels.mean(axis=0)
    std_color = segment_pixels.std(axis=0)
    
    # Get bounding box for shape features
    y_indices, x_indices = np.where(mask)
    if len(y_indices) == 0:
        return None, 0.0
    
    x_min, x_max = x_indices.min(), x_indices.max()
    y_min, y_max = y_indices.min(), y_indices.max()
    
    width = x_max - x_min
    height = y_max - y_min
    aspect_ratio = width / max(height, 1)
    area = len(segment_pixels)
    
    r, g, b = mean_color
    
    # Heuristic classification rules
    # Cracks: Dark, elongated
    if r < 80 and g < 80 and b < 80 and aspect_ratio > 4:
        return 1, 0.75  # cracks
    
    # Water damage: Brownish discoloration
    if r > 100 and r > g * 1.2 and r > b * 1.3:
        return 3, 0.70  # water_damage
    
    # Mold: Dark greenish/black spots
    if r < 90 and g < 90 and b < 90 and std_color.mean() < 30:
        return 2, 0.68  # mold
    
    # Structural damage: Large, irregular
    if area > 5000 and aspect_ratio < 2:
        return 4, 0.60  # structural_damage
    
    # Wall damage: Medium-sized irregular areas
    if area > 1000 and area < 5000:
        return 11, 0.55  # wall_damage
    
    # Floor damage: Low in image, horizontal
    if y_min > image.shape[0] * 0.6 and aspect_ratio > 2:
        return 10, 0.50  # floor_damage
    
    # Ceiling damage: High in image
    if y_max < image.shape[0] * 0.4:
        return 12, 0.50  # ceiling_damage
    
    # Default: general damage
    return 0, 0.45  # general_damage

print("✅ Defect classifier defined")
print(f"   Classes: {len(DEFECT_CLASSES)}")
print(f"   Method: Enhanced heuristic-based (color, shape, texture, position)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 5: Extract and Split Images into Batches
# ═══════════════════════════════════════════════════════════════

import zipfile
from pathlib import Path
import shutil

# Find ZIP file
ZIP_PATH = None
for path in ZIP_PATHS:
    if os.path.exists(path):
        ZIP_PATH = path
        break

if not ZIP_PATH:
    print("❌ ZIP file not found!")
    print("\nPlease upload 'filtered_images.zip' to one of these locations:")
    for path in ZIP_PATHS:
        print(f"  - {path}")
    raise FileNotFoundError("ZIP file not found")

print(f"✅ Found ZIP: {ZIP_PATH}")
print(f"   Size: {os.path.getsize(ZIP_PATH) / 1024**2:.1f} MB")

# Extract all images first
EXTRACT_DIR = '/content/all_images'
os.makedirs(EXTRACT_DIR, exist_ok=True)

print("\n📦 Extracting images...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

# Find all images
image_files = []
for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
    image_files.extend(list(Path(EXTRACT_DIR).rglob(ext)))

total_images = len(image_files)
print(f"✅ Found {total_images:,} images in ZIP")

# Limit to MAX_IMAGES
if total_images > MAX_IMAGES:
    print(f"⚠️  Limiting to {MAX_IMAGES:,} images (found {total_images:,})")
    image_files = image_files[:MAX_IMAGES]
else:
    print(f"✅ Processing all {total_images:,} images")

# Split into batches
print(f"\n📂 Splitting into {NUM_BATCHES} batches of {IMAGES_PER_BATCH} images each...")

batches = []
for i in range(NUM_BATCHES):
    start_idx = i * IMAGES_PER_BATCH
    end_idx = min(start_idx + IMAGES_PER_BATCH, len(image_files))
    batch_images = image_files[start_idx:end_idx]
    batches.append(batch_images)
    print(f"   Batch {i+1}: {len(batch_images):,} images (indices {start_idx} to {end_idx-1})")

print(f"\n✅ Dataset split into {len(batches)} batches")
print(f"   Total images to process: {sum(len(b) for b in batches):,}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 6: Auto-Label All Images (Batch Processing)
# ═══════════════════════════════════════════════════════════════

from tqdm import tqdm
import json
from datetime import datetime

# Create output directory structure
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Global statistics
global_stats = {
    'total_images': sum(len(b) for b in batches),
    'processed': 0,
    'images_with_defects': 0,
    'total_detections': 0,
    'detections_per_class': {name: 0 for name in CLASS_NAMES},
    'batches': [],
    'failed_images': [],
    'start_time': datetime.now().isoformat(),
}

print("="*70)
print("🚀 Starting SAM 2 Auto-Labeling")
print("="*70)
print(f"Total images: {global_stats['total_images']:,}")
print(f"Batches: {len(batches)}")
print(f"Output: {OUTPUT_DIR}")
print(f"\n⏳ Processing (estimated time: {global_stats['total_images'] / 500:.1f} hours on T4 GPU)...\n")

# Process each batch
for batch_num, batch_images in enumerate(batches, 1):
    print(f"\n{'='*70}")
    print(f"📦 Processing Batch {batch_num}/{len(batches)} ({len(batch_images):,} images)")
    print(f"{'='*70}\n")
    
    # Create batch directories
    batch_dir = os.path.join(OUTPUT_DIR, f'batch_{batch_num}')
    batch_images_dir = os.path.join(batch_dir, 'images')
    batch_labels_dir = os.path.join(batch_dir, 'labels')
    
    os.makedirs(batch_images_dir, exist_ok=True)
    os.makedirs(batch_labels_dir, exist_ok=True)
    
    batch_stats = {
        'batch_num': batch_num,
        'total_images': len(batch_images),
        'processed': 0,
        'images_with_defects': 0,
        'total_detections': 0,
        'detections_per_class': {name: 0 for name in CLASS_NAMES},
        'failed_images': []
    }
    
    # Process images in this batch
    for image_path in tqdm(batch_images, desc=f"Batch {batch_num}"):
        try:
            # Load image
            image = cv2.imread(str(image_path))
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            height, width = image_rgb.shape[:2]
            
            # Generate masks automatically
            masks = mask_generator.generate(image_rgb)
            
            # Filter and classify masks
            valid_detections = []
            
            for mask_data in masks:
                mask = mask_data['segmentation']
                
                # Classify the segment
                class_id, confidence = classify_segment_heuristic(mask, image_rgb)
                
                if class_id is None or confidence < 0.4:
                    continue
                
                # Get bounding box from mask
                y_indices, x_indices = np.where(mask)
                if len(y_indices) == 0:
                    continue
                
                x_min = x_indices.min()
                x_max = x_indices.max()
                y_min = y_indices.min()
                y_max = y_indices.max()
                
                bbox_width = x_max - x_min
                bbox_height = y_max - y_min
                
                # Filter tiny detections
                if bbox_width < 10 or bbox_height < 10:
                    continue
                
                valid_detections.append({
                    'class_id': class_id,
                    'bbox': [x_min, y_min, bbox_width, bbox_height],
                    'confidence': confidence
                })
            
            # Save image and labels if detections found
            if valid_detections:
                image_id = image_path.stem
                
                # Copy image to batch directory
                dest_image_path = os.path.join(batch_images_dir, f"{image_id}{image_path.suffix}")
                shutil.copy2(str(image_path), dest_image_path)
                
                # Save YOLO format labels
                label_path = os.path.join(batch_labels_dir, f"{image_id}.txt")
                
                with open(label_path, 'w') as f:
                    for det in valid_detections:
                        x, y, w, h = det['bbox']
                        class_id = det['class_id']
                        
                        # YOLO format: class x_center y_center width height (normalized)
                        x_center = (x + w / 2) / width
                        y_center = (y + h / 2) / height
                        w_norm = w / width
                        h_norm = h / height
                        
                        f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")
                
                # Update statistics
                batch_stats['images_with_defects'] += 1
                batch_stats['total_detections'] += len(valid_detections)
                global_stats['images_with_defects'] += 1
                global_stats['total_detections'] += len(valid_detections)
                
                for det in valid_detections:
                    class_name = CLASS_NAMES[det['class_id']]
                    batch_stats['detections_per_class'][class_name] += 1
                    global_stats['detections_per_class'][class_name] += 1
            
            batch_stats['processed'] += 1
            global_stats['processed'] += 1
            
        except Exception as e:
            error_msg = f"{image_path}: {str(e)}"
            batch_stats['failed_images'].append(error_msg)
            global_stats['failed_images'].append(error_msg)
            continue
    
    # Save batch statistics
    batch_stats_file = os.path.join(batch_dir, 'batch_stats.json')
    with open(batch_stats_file, 'w') as f:
        json.dump(batch_stats, f, indent=2)
    
    global_stats['batches'].append(batch_stats)
    
    # Print batch summary
    print(f"\n✅ Batch {batch_num} Complete!")
    print(f"   Processed: {batch_stats['processed']:,} / {batch_stats['total_images']:,}")
    print(f"   Images with defects: {batch_stats['images_with_defects']:,}")
    print(f"   Total detections: {batch_stats['total_detections']:,}")
    print(f"   Failed: {len(batch_stats['failed_images'])}")

# Save global statistics
global_stats['end_time'] = datetime.now().isoformat()
global_stats_file = os.path.join(OUTPUT_DIR, 'global_stats.json')
with open(global_stats_file, 'w') as f:
    json.dump(global_stats, f, indent=2)

# Create data.yaml for YOLO training
data_yaml = f"""path: {OUTPUT_DIR}
train: batch_1/images
val: batch_2/images

nc: {len(DEFECT_CLASSES)}
names: {CLASS_NAMES}
"""

with open(os.path.join(OUTPUT_DIR, 'data.yaml'), 'w') as f:
    f.write(data_yaml)

print("\n" + "="*70)
print("✅ AUTO-LABELING COMPLETE!")
print("="*70)
print(f"\nTotal processed: {global_stats['processed']:,} / {global_stats['total_images']:,}")
print(f"Images with defects: {global_stats['images_with_defects']:,}")
print(f"Total detections: {global_stats['total_detections']:,}")
print(f"Failed: {len(global_stats['failed_images'])}")

print("\n📊 Detections per class (global):")
for class_name, count in sorted(global_stats['detections_per_class'].items(), 
                                key=lambda x: x[1], reverse=True):
    if count > 0:
        print(f"   {class_name:20s}: {count:,}")

print(f"\n📁 Output structure:")
for i in range(1, NUM_BATCHES + 1):
    batch_dir = f"batch_{i}"
    images_count = len(list(Path(os.path.join(OUTPUT_DIR, batch_dir, 'images')).glob('*')))
    labels_count = len(list(Path(os.path.join(OUTPUT_DIR, batch_dir, 'labels')).glob('*.txt')))
    print(f"   {batch_dir}/ → {images_count:,} images, {labels_count:,} labels")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 7: Package and Download Results Immediately
# ═══════════════════════════════════════════════════════════════

import shutil
from google.colab import files

print("="*70)
print("📦 Packaging Results")
print("="*70)

# Create ZIP file
OUTPUT_ZIP = '/content/sam2_labeled_2000_images.zip'
print(f"\n🔄 Creating ZIP archive...")
shutil.make_archive('/content/sam2_labeled_2000_images', 'zip', OUTPUT_DIR)

zip_size_mb = os.path.getsize(OUTPUT_ZIP) / (1024**2)
print(f"✅ ZIP created: {zip_size_mb:.1f} MB")

# Save to Google Drive
drive_output_dir = '/content/drive/MyDrive/SAM2_AutoLabel/'
os.makedirs(drive_output_dir, exist_ok=True)

drive_output_file = os.path.join(drive_output_dir, 'sam2_labeled_2000_images.zip')
print(f"\n💾 Copying to Google Drive...")
shutil.copy2(OUTPUT_ZIP, drive_output_file)
print(f"✅ Saved to Drive: {drive_output_file}")

# Download immediately if configured
if DOWNLOAD_IMMEDIATELY:
    print(f"\n⬇️  Starting automatic download...")
    print(f"   File: sam2_labeled_2000_images.zip ({zip_size_mb:.1f} MB)")
    print(f"   This may take a few minutes...\n")
    
    files.download(OUTPUT_ZIP)
    
    print("\n✅ Download started!")
    print("   Check your browser's download folder")
else:
    print("\n⚠️  Auto-download disabled (DOWNLOAD_IMMEDIATELY=False)")
    print("   You can manually download from Google Drive:")
    print(f"   {drive_output_file}")

print("\n" + "="*70)
print("🎉 ALL DONE!")
print("="*70)
print("\n📋 Summary:")
print(f"   ✅ Processed: {global_stats['processed']:,} images")
print(f"   ✅ Split into: {NUM_BATCHES} batches")
print(f"   ✅ Total detections: {global_stats['total_detections']:,}")
print(f"   ✅ ZIP size: {zip_size_mb:.1f} MB")
print(f"   ✅ Saved to Drive: Yes")
print(f"   ✅ Downloaded: {'Yes' if DOWNLOAD_IMMEDIATELY else 'No'}")

print("\n📁 Dataset ready for YOLO training!")
print("   Use batch_1 for training, batch_2 for validation")